In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
import os

import pathlib as pl

import math
import tifffile

In [ ]:
import matplotlib
matplotlib.rcParams['svg.fonttype'] = 'none'

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

import torch
print(torch.cuda.get_device_name(0))

# Import and preprocess data

## Helper functions

In [ ]:
def build_palettes_from_adata(adata, palette_specs):
    """
    Build labeled color palettes for categorical columns in adata.obs.

    Parameters
    ----------
    adata : AnnData
        Must have .obs DataFrame containing categorical columns.
    palette_specs : dict
        Mapping {column_name: palette} where palette can be:
          - a string palette name (e.g. "tab10")
          - a list of RGB colors (custom)

    Returns
    -------
    dict
        {column_name: {label: color}} mapping.
    """
    custom_palettes = {}

    for col, palette in palette_specs.items():
        if col not in adata.obs.columns:
            print(f"⚠️ Warning: '{col}' not found in adata.obs — skipping.")
            continue

        unique_vals = sorted(adata.obs[col].astype(str).dropna().unique())
        n_unique = len(unique_vals)

        # If user passed a name → generate via seaborn
        if isinstance(palette, str):
            pal_colors = sns.color_palette(palette, n_colors=n_unique)
        # If user passed a list → use directly
        elif isinstance(palette, (list, tuple)):
            pal_colors = palette[:n_unique]
        else:
            raise ValueError(f"Unsupported palette type for '{col}': {type(palette)}")

        color_dict = dict(zip(unique_vals, pal_colors))
        custom_palettes[col] = color_dict

    print(f"✅ Built palettes for {len(custom_palettes)} columns.")
    return custom_palettes


In [ ]:
def plot_celltype_spatial_single_split_legend(
    df,
    color_by="celltype",
    sample_id=None,
    title=None,
    palette_dict=None,         # ✅ added
    palette_name="tab20",
    s=1.5,
    save_svg=True,
    output_prefix="spatial_plot",
    legend_title=None,
):
    """
    Nature Genetics–style spatial scatterplot for one sample,
    saving main plot as PNG (raster) and legend separately as SVG (vector).
    """
    sns.set_style("white")
    sns.set_context("talk")

    # --- Subset one sample ---
    if sample_id is not None:
        df = df[df["sample_id"] == sample_id].copy()
        if df.empty:
            raise ValueError(f"Sample ID '{sample_id}' not found in DataFrame.")

    # --- Colors ---
    unique_labels = sorted(df[color_by].dropna().unique())
    if palette_dict is not None and color_by in palette_dict:
        color_dict = palette_dict[color_by]
    else:
        palette = sns.color_palette(palette_name, n_colors=len(unique_labels))
        color_dict = dict(zip(unique_labels, palette))

    # --- Main plot ---
    fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
    sns.scatterplot(
        data=df,
        x="X_coord", y="Y_coord",
        hue=color_by, palette=color_dict,
        s=s, alpha=0.9, linewidth=0,
        rasterized=True, ax=ax, legend=False
    )
    ax.invert_yaxis(); ax.set_aspect("equal", adjustable="box")
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(""); ax.set_ylabel("")
    plt.tight_layout()

    # --- Save main figure ---
    fname_main = f"{output_prefix}_{sample_id or 'sample'}_main.png"
    fig.savefig(fname_main, dpi=300, bbox_inches="tight", transparent=True, format="png")
    print(f"Saved main figure: {fname_main}")

    # --- Legend ---
    fig_leg, ax_leg = plt.subplots(figsize=(3, 0.5 * len(unique_labels)), dpi=300)
    handles = [
        plt.Line2D([0], [0], marker='o', color='none', label=label,
                   markerfacecolor=color_dict[label], markersize=8)
        for label in unique_labels
    ]
    ax_leg.legend(handles=handles, loc="center left", frameon=False,
                  title=legend_title or color_by, title_fontsize=14, fontsize=14)
    ax_leg.axis("off")
    plt.tight_layout()

    if save_svg:
        fname_leg = f"{output_prefix}_{sample_id or 'sample'}_legend.svg"
        fig_leg.savefig(fname_leg, dpi=300, bbox_inches="tight", transparent=True, format="svg")
        print(f"Saved legend: {fname_leg}")

    plt.close(fig); plt.close(fig_leg)


## Analysis

In [ ]:
rawdata = sc.read_h5ad('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/adata.h5ad')

In [ ]:
rawdata.obs[['major_celltype','minor_celltype','cell_id']].to_csv('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/celltypes_brca.csv')

In [ ]:
rawdata = sc.read_h5ad('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/adata.h5ad')

region_annot = pd.read_csv('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/breast_ffpe_region_annotations.csv',index_col=0)
region_annot.index = region_annot.index.astype(str)

rawdata.obs['path_region'] = region_annot.loc[rawdata.obs_names].values.ravel()

rawdata.obs = pd.concat([rawdata.obs, pd.DataFrame(rawdata.obsm['spatial_px'], index=rawdata.obs_names, columns=['X_coord','Y_coord'])],axis=1)

region_df = rawdata.obs[['minor_celltype', 'major_celltype', 'cell_id',
       'path_region', 'X_coord','Y_coord']]

region_df['sample_id'] = 'TENXBreast'

In [ ]:
tab_filtered = sns.color_palette()
tab_filtered = [c for i,c in enumerate(tab_filtered) if i not in [4,6]]

tab20_filtered = sns.color_palette('tab20') + sns.color_palette('tab20c')[:11]
tab20_filtered = [c for i,c in enumerate(tab20_filtered) if i not in [8,9,12,13]]

In [ ]:
palette_specs = {
            "path_region": tab_filtered,
            "major_celltype": tab_filtered,
            "minor_celltype": tab20_filtered,
        }

palette_dict_1 = build_palettes_from_adata(rawdata, palette_specs)

In [ ]:
cpal = sns.color_palette()
palette_dict_1['major_celltype'] = {'Malignant': cpal[0], 'Pre-malignant': cpal[1], 'Immune': cpal[2],
                  'Myoepithelial': cpal[3], 'Stromal': cpal[5], 'Endothelial': cpal[9],
                  'Other': cpal[8], 'Unlabeled': cpal[7]}

In [ ]:
plot_celltype_spatial_single_split_legend(
    region_df,
    color_by="major_celltype",
    sample_id=None,
    title='BRCA\nCell type',
    palette_dict=palette_dict_1,
    s=1.5,
    save_svg=True,
    output_prefix="../../../SpatialFusion/results/figures_Fig3/BRCA_celltype",
    legend_title='Cell Type'
)

In [ ]:
plot_celltype_spatial_single_split_legend(
    region_df,
    color_by="minor_celltype",
    sample_id=None,
    title='BRCA\nCell subtype',
    palette_dict=palette_dict_1,
    s=1.5,
    save_svg=True,
    output_prefix="../../../SpatialFusion/results/figures_Fig3/BRCA_cellsubtype",
    legend_title='Cell Subtype'
)

In [ ]:
plot_celltype_spatial_single_split_legend(
    region_df,
    color_by="path_region",
    sample_id=None,
    title='BRCA\nPathologist-annotated region',
    palette_dict=palette_dict_1,
    s=1.5,
    save_svg=True,
    output_prefix="../../../SpatialFusion/results/figures_Fig3/BRCA_pathregion",
    legend_title='Path.-annotated region'
)

## Download estimated pathways

# Embed sample

In [ ]:
from spatialfusion.embed.embed import AEInputs, run_full_embedding

In [ ]:
basepath = pl.Path('../../../Broad_SpatialFoundation/test_data/')
sample_name = '10X_Xenium_Breast_FFPE'
output_dir = basepath / sample_name

adata = sc.read_h5ad(basepath / sample_name / 'adata.h5ad')
adata.obs = pd.concat([adata.obs, pd.DataFrame(adata.obsm['spatial_px'], index=adata.obs_names, columns=['X_coord','Y_coord'])],axis=1)
adata.obs["sample_id"] = sample_name

pathway_matrix = pd.read_parquet(basepath / sample_name / 'pathway_activation.parquet')

pathways_by_sample = {
    sample_name: pathway_matrix,
}

In [ ]:
uni_df = pd.read_parquet(pl.Path(output_dir) / 'embeddings' / 'UNI.parquet')
scgpt_df = pd.read_parquet(pl.Path(output_dir) / 'embeddings' / 'scGPT.parquet')
virchow_df = pd.read_parquet(pl.Path(output_dir) / 'embeddings' / 'Virchow2.parquet')
nicheformer_df = pd.read_parquet(pl.Path(output_dir) / 'embeddings' / 'nicheformer.parquet')

## Embedding UNI2+scGPT

In [ ]:
ae_inputs_by_sample = {
    sample_name: AEInputs(adata=adata, z_he=uni_df, z_rna=scgpt_df),
}

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/uni_scgpt_full_20260602-024030_1fbc1fe4/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_fusion_sweep/uni_scgpt_full_gcn_avg_reg_cls_836df169/model.pt',
    device="cuda:0",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
embeddings_df.index = embeddings_df.cell_id

out_path = "../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/GCN_embeddings_final/uni_scgpt_avg.parquet"
embeddings_df.to_parquet(out_path)

## Embedding UNI2+Nicheformer

In [ ]:
ae_inputs_by_sample = {
    sample_name: AEInputs(adata=adata, z_he=uni_df, z_rna=nicheformer_df),
}

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/uni_nicheformer_full_20260602-051319_a4b53581/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_fusion_sweep/uni_nicheformer_full_gcn_avg_reg_cls_80d67272/model.pt',
    device="cuda:0",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
embeddings_df.index = embeddings_df.cell_id

out_path = "../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/GCN_embeddings_final/uni_nicheformer_avg.parquet"
embeddings_df.to_parquet(out_path)

## Embedding Virchow+scGPT

In [ ]:
ae_inputs_by_sample = {
    sample_name: AEInputs(adata=adata, z_he=virchow_df, z_rna=scgpt_df),
}

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/virchow_scgpt_full_20260602-065430_e436487d/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_fusion_sweep/virchow_scgpt_full_gcn_avg_reg_cls_6c8c309d/model.pt',
    device="cuda:0",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
embeddings_df.index = embeddings_df.cell_id

out_path = "../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/GCN_embeddings_final/virchow_scgpt_avg.parquet"
embeddings_df.to_parquet(out_path)

## Embedding Virchow+Nicheformer

In [ ]:
ae_inputs_by_sample = {
    sample_name: AEInputs(adata=adata, z_he=virchow_df, z_rna=nicheformer_df),
}

In [ ]:
# this uses the average version
embeddings_df = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/virchow_nicheformer_full_20260602-084626_531b291c/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_fusion_sweep/virchow_nicheformer_full_gcn_avg_reg_cls_05276acb/model.pt',
    device="cuda:0",
    combine_mode="average",
    spatial_key='spatial_px',
    celltype_key='major_celltype',
    save_ae_dir=None,  # optional
)

In [ ]:
embeddings_df.index = embeddings_df.cell_id

out_path = "../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/GCN_embeddings_final/virchow_nicheformer_avg.parquet"
embeddings_df.to_parquet(out_path)

# SDMBench

## Helper functions

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

import scanpy as sc

from scipy.spatial import distance
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *


In [ ]:
def compute_PAS_fast(clusterlabel, location, k=10):
    clusterlabel = np.array(clusterlabel)
    location = np.array(location)

    # Fit NearestNeighbors (ignore self-match later)
    nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto').fit(location)
    distances, indices = nbrs.kneighbors(location)

    # Remove self (first column is self in most cases)
    neighbor_indices = indices[:, 1:]  # shape: (n_samples, k)

    # Check PAS condition
    mismatches = np.array([
        np.sum(clusterlabel[neighbor_indices[i]] != clusterlabel[i]) > (k / 2)
        for i in range(len(clusterlabel))
    ])

    return np.sum(mismatches) / len(clusterlabel)


def compute_CHAOS_fast(clusterlabel, location):
    clusterlabel = np.array(clusterlabel)
    location = np.array(location)
    matched_location = StandardScaler().fit_transform(location)

    clusterlabel_unique = np.unique(clusterlabel)
    dist_val = 0
    total_count = 0

    for k in tqdm(clusterlabel_unique, desc="Computing CHAOS"):
        cluster_mask = clusterlabel == k
        location_cluster = matched_location[cluster_mask]
        n = location_cluster.shape[0]

        if n <= 2:
            continue

        # Use NearestNeighbors to find 1-NN distances
        nbrs = NearestNeighbors(n_neighbors=2, algorithm='auto').fit(location_cluster)
        distances, _ = nbrs.kneighbors(location_cluster)

        # distances[:, 0] is zero (self), distances[:, 1] is nearest neighbor
        dist_val += np.sum(distances[:, 1])
        total_count += n

    return dist_val / total_count if total_count > 0 else np.nan


def compute_ASW_fast(adata, pred_key, spatial_key='spatial'):
    coords = adata.obsm[spatial_key]
    labels = adata.obs[pred_key]
    return silhouette_score(X=coords, labels=labels, metric='euclidean')

def compute_ARI(adata,gt_key,pred_key):
        return adjusted_rand_score(adata.obs[gt_key],adata.obs[pred_key])

def compute_NMI(adata,gt_key,pred_key):
    return normalized_mutual_info_score(adata.obs[gt_key],adata.obs[pred_key])

def compute_HOM(adata,gt_key,pred_key):
    return homogeneity_score(adata.obs[gt_key],adata.obs[pred_key])

def compute_COM(adata,gt_key,pred_key):
    return completeness_score(adata.obs[gt_key],adata.obs[pred_key])

## Analysis

In [ ]:
adata = rawdata.copy()

In [ ]:
emb_uni_scgpt_df = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/GCN_embeddings_final/uni_scgpt_avg.parquet')

emb_virchow_scgpt_df = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/GCN_embeddings_final/virchow_scgpt_avg.parquet')

emb_uni_nicheformer_df = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/GCN_embeddings_final/uni_nicheformer_avg.parquet')

emb_virchow_nicheformer_df = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/GCN_embeddings_final/virchow_nicheformer_avg.parquet')

In [ ]:
emb_he_df = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/embeddings/NicheFinder_he_new.parquet')

emb_finetuned_df = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/embeddings/NicheFinder_finetuned_new.parquet')

In [ ]:
#banksy_embeddings = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/embeddings/banksy.parquet')
banksy_08_embeddings = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/embeddings/banksy_08.parquet')

nichecompass_embeddings = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/embeddings/nichecompass.parquet')

nicheformer_embeddings = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/embeddings/nicheformer.parquet')

scgptspatial_embeddings = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/embeddings/scGPTspatial.parquet')

omiclip_text_embeddings = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/OmiCLIP_text_emb.parquet')

omiclip_image_embeddings = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/OmiCLIP_image_emb.parquet')

In [ ]:
emb_uni_scgpt_df.columns = emb_uni_scgpt_df.columns.astype(str)

emb_virchow_scgpt_df.columns = emb_virchow_scgpt_df.columns.astype(str)

emb_uni_nicheformer_df.columns = emb_uni_nicheformer_df.columns.astype(str)

emb_virchow_nicheformer_df.columns = emb_virchow_nicheformer_df.columns.astype(str)

In [ ]:
adata.obsm['gcn_uni_scgpt'] = emb_uni_scgpt_df.loc[adata.obs_names,['0','1','2','3','4','5','6','7','8','9']]

adata.obsm['gcn_virchow_scgpt'] = emb_virchow_scgpt_df.loc[adata.obs_names,['0','1','2','3','4','5','6','7','8','9']]

adata.obsm['gcn_uni_nicheformer'] = emb_uni_nicheformer_df.loc[adata.obs_names,['0','1','2','3','4','5','6','7','8','9']]

adata.obsm['gcn_virchow_nicheformer'] = emb_virchow_nicheformer_df.loc[adata.obs_names,['0','1','2','3','4','5','6','7','8','9']]

In [ ]:
adata.obsm['gcn_he'] = emb_he_df.loc[adata.obs_names,['0','1','2','3','4','5','6','7','8','9']]

adata.obsm['gcn_finetuned'] = emb_finetuned_df.loc[adata.obs_names,['0','1','2','3','4','5','6','7','8','9']]

In [ ]:
#adata.obsm['banksy'] = banksy_embeddings.loc[adata.obs_names]
adata.obsm['banksy_08'] = banksy_08_embeddings.loc[adata.obs_names]

adata.obsm['nichecompass'] = nichecompass_embeddings.loc[adata.obs_names]

adata.obsm['nicheformer'] = nicheformer_embeddings.loc[adata.obs_names]

adata.obsm['scgptspatial'] = scgptspatial_embeddings.loc[adata.obs_names]

adata.obsm['omiclip_text'] = omiclip_text_embeddings.loc[adata.obs_names]

adata.obsm['omiclip_image'] = omiclip_image_embeddings.loc[adata.obs_names]

## Run clustering

The resolutions are set here to have the closest amount to the set number of clusters (here, 11). Small clusters are grouped together so that all methods have the same amount of clusters for comparison.

In [ ]:
# When rerunning after intermediate results
adata_obs= pd.read_csv('benchmark_breast_adata_obs.csv',index_col=0)
adata_obs.index = adata_obs.index.astype(str)

adata.obs = adata_obs

In [ ]:
sc.pp.neighbors(adata, use_rep = 'gcn_uni_scgpt')

In [ ]:
sc.tl.leiden(adata, resolution=0.15, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_gcn_uni_scgpt'] = adata.obs.leiden.replace({'11': '10', '12': '10', '13': '10', '14': '10', '15': '10',
                                                             '16': '10', '17': '10',})

In [ ]:
sc.pp.neighbors(adata, use_rep = 'gcn_uni_nicheformer')

In [ ]:
sc.tl.leiden(adata, resolution=0.15, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_gcn_uni_nicheformer'] = adata.obs.leiden

In [ ]:
sc.pp.neighbors(adata, use_rep = 'gcn_virchow_scgpt')

In [ ]:
sc.tl.leiden(adata, resolution=0.145, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_gcn_virchow_scgpt'] = adata.obs.leiden.replace({'11': '10', '12': '10', '13': '10', })

In [ ]:
sc.pp.neighbors(adata, use_rep = 'gcn_virchow_nicheformer')

In [ ]:
sc.tl.leiden(adata, resolution=0.16, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_gcn_virchow_nicheformer'] = adata.obs.leiden.replace({'11': '10', '12': '10', '13': '10', '14': '10', })

In [ ]:
sc.pp.neighbors(adata, use_rep = 'gcn_finetuned_uni_scgpt')

In [ ]:
sc.tl.leiden(adata, resolution=0.14, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_gcn_finetuned_uni_scgpt'] = adata.obs.leiden.replace({'11': '10', '12': '10', '13': '10',})

In [ ]:
sc.pp.neighbors(adata, use_rep = 'gcn_finetuned_virchow_scgpt')

In [ ]:
sc.tl.leiden(adata, resolution=0.15, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_gcn_finetuned_virchow_scgpt'] = adata.obs.leiden.replace({'11': '10',})

In [ ]:
sc.pp.neighbors(adata, use_rep = 'nicheformer')

In [ ]:
sc.tl.leiden(adata, resolution=0.9, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_nicheformer'] = adata.obs.leiden

In [ ]:
sc.pp.neighbors(adata, use_rep = 'gcn_he')

In [ ]:
sc.tl.leiden(adata, resolution=0.12, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_gcn_he'] = adata.obs.leiden

In [ ]:
sc.pp.neighbors(adata, use_rep = 'gcn_finetuned')

In [ ]:
sc.tl.leiden(adata, resolution=0.15, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_gcn_finetuned'] = adata.obs.leiden

In [ ]:
sc.pp.neighbors(adata, use_rep='nichecompass')

sc.tl.leiden(adata, resolution=0.3)

adata.obs['leiden_nichecompass'] = adata.obs.leiden

sc.pp.neighbors(adata, use_rep='nicheformer')

sc.tl.leiden(adata, resolution=0.75)

adata.obs['leiden_nicheformer'] = adata.obs.leiden

sc.pp.neighbors(adata, use_rep='scgptspatial')

sc.tl.leiden(adata, resolution=0.32)

adata.obs['leiden_scgptspatial'] = adata.obs.leiden

adata.obs['leiden_scgptspatial'] = adata.obs['leiden_scgptspatial'].replace({'11': '10'})

sc.pp.neighbors(adata, use_rep='omiclip_text')

sc.tl.leiden(adata, resolution=0.4)

adata.obs['leiden_omiclip_text'] = adata.obs.leiden

sc.pp.neighbors(adata, use_rep='omiclip_image')

sc.tl.leiden(adata, resolution=0.25)

adata.obs['leiden_omiclip_image'] = adata.obs.leiden.replace({'11': '10'})

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

sc.tl.pca(adata)

sc.pp.neighbors(adata, use_rep='X_pca')

sc.tl.leiden(adata, resolution=0.9)

adata.obs['leiden_scanpy'] = adata.obs.leiden

In [ ]:
sc.pp.neighbors(adata, use_rep = 'banksy_08')

In [ ]:
sc.tl.leiden(adata, resolution=0.4, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_banksy_08'] = adata.obs.leiden.replace({'11': '10', '12': '10',})

### NOVAE WAS RUN WITH ITS INTERNAL MECHANISM FOR DOMAIN DISCOVERY

In [ ]:
novae = pd.read_parquet('../../../Broad_SpatialFoundation/test_data/10X_Xenium_Breast_FFPE/embeddings/novae_11.parquet')

In [ ]:
novae = novae.novae_domains_11.cat.codes

In [ ]:
novae.value_counts()

In [ ]:
adata.obs['leiden_novae'] = novae.loc[adata.obs_names]

In [ ]:
adata.obs.to_csv('benchmark_breast_adata_obs.csv')

## Compute metrics

In [ ]:
def compute_all_metrics(adata, clustering_keys, ground_truth_key='path_region', spatial_key='spatial_px'):
    results = {}

    for method_name, cluster_key in clustering_keys.items():
        metrics = {
            'ARI': compute_ARI(adata, cluster_key, ground_truth_key),
            'NMI': compute_NMI(adata, cluster_key, ground_truth_key),
            'HOM': compute_HOM(adata, cluster_key, ground_truth_key),
            'COM': compute_COM(adata, cluster_key, ground_truth_key),
            'PAS': compute_PAS_fast(adata.obs[cluster_key], adata.obsm[spatial_key]),
            'CHAOS': compute_CHAOS_fast(adata.obs[cluster_key], adata.obsm[spatial_key]),
        }
        results[method_name] = metrics

    return pd.DataFrame(results)

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

def format_number(value):
    """Format numbers: scientific notation if <0.01, else 2 decimals."""
    if pd.isna(value):
        return ""
    if abs(value) < 0.01 and value != 0:
        return f"{value:.0e}"  # 1 decimal in scientific notation, e.g. 3.4e-04
    else:
        return f"{value:.2f}"  # two decimals otherwise

def plot_benchmark_heatmap(
    results_df,
    title="Spatial clustering benchmark",
    savefig=None,
    metric_order=None,
):
    """
    Nature Genetics–style benchmarking heatmap showing method rankings across metrics.
    Allows manual control of metric order.
    """

    lower_better = {'PAS', 'CHAOS'}

    # --- Default metric order ---
    if metric_order is None:
        metric_order = list(results_df.index)

    # --- Normalize scores ---
    df_norm = results_df.copy()
    for metric in df_norm.index:
        vals = df_norm.loc[metric]
        if metric in lower_better:
            vals = -vals
        df_norm.loc[metric] = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)

    # --- Rank per metric ---
    ranks = results_df.copy()
    for metric in ranks.index:
        ranks.loc[metric] = results_df.loc[metric].rank(ascending=(metric in lower_better))

    # --- Prepare longform for plotting ---
    df_plot = df_norm.reset_index().melt(
        id_vars='index', var_name='Method', value_name='Normalized'
    ).rename(columns={'index': 'Metric'})

    df_plot['Raw'] = results_df.reset_index().melt(
        id_vars='index', var_name='Method', value_name='Raw'
    )['Raw']

    df_plot['Rank'] = ranks.reset_index().melt(
        id_vars='index', var_name='Method', value_name='Rank'
    )['Rank']

    # Add directional arrows
    df_plot['MetricLabel'] = df_plot['Metric'].apply(
        lambda m: f"{m} {'↓' if m in lower_better else '↑'}"
    )

    # --- Construct ordered MetricLabel list ---
    metric_order_labels = []
    for m in metric_order:
        arrow = '↓' if m in lower_better else '↑'
        metric_order_labels.append(f"{m} {arrow}")

    # --- Heatmap data matrix ---
    method_order = results_df.columns.tolist()
    df_matrix = df_plot.pivot_table(
        index="MetricLabel", columns="Method", values="Normalized"
    ).loc[metric_order_labels, method_order]

    # --- Aesthetics ---
    sns.set_theme(style="white", context="talk")

    fig, ax = plt.subplots(figsize=(1.3 * len(method_order), 0.8 * len(metric_order)), dpi=300)
    # Enhance contrast near the top (gamma correction)
    gamma = 3  ### THIS IS ONLY FOR THE COLOR FOR PLOTTING PURPOSES, NOT THE NUMBERS!
    df_matrix_contrast = df_matrix ** gamma
    sns.heatmap(
        df_matrix_contrast,
        #cmap="vlag",
        cmap = LinearSegmentedColormap.from_list(
            "vlag_red",
            ["#fee8ef",  # very light pink
             "#f4a3a8",  # pastel red
             "#d95858",  # mid red
             "#b40426"]  # vlag red (vivid crimson)
        ),
        cbar=False,
        ax=ax,
        linewidths=0,
        square=True,
    )

    # --- Adaptive text color (white on dark, black on light) ---
    #cmap = plt.get_cmap("vlag")
    cmap = LinearSegmentedColormap.from_list(
        "vlag_red",
        ["#fee8ef",  # very light pink
         "#f4a3a8",  # pastel red
         "#d95858",  # mid red
         "#b40426"]  # vlag red (vivid crimson)
    )

    for i, metric in enumerate(df_matrix.index):
        base_metric = metric.split()[0]
        for j, method in enumerate(df_matrix.columns):
            raw_val = results_df.loc[base_metric, method]
            norm_val = df_matrix.loc[metric, method]

            # Compute luminance for adaptive color
            rgb = np.array(cmap(norm_val)[:3])
            luminance = 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]
            text_color = "black" if luminance > 0.5 else "white"

            ax.text(
                j + 0.5, i + 0.5,
                format_number(raw_val),
                ha='center', va='center',
                color=text_color,
                fontsize=8,
                fontweight='normal',
            )

    # --- Formatting ---
    ax.set_title(title, fontsize=10, pad=14, fontweight='normal')
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=10, fontweight='normal')
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=10, fontweight='normal')

    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()

    if savefig:
        fig.savefig(
            savefig,
            bbox_inches="tight",
            dpi=300,
            format=savefig.split('.')[-1],
            transparent=True
        )
        print(f"Saved: {savefig}")

    plt.show()



In [ ]:
adata_obs= pd.read_csv('benchmark_breast_adata_obs.csv',index_col=0)
adata_obs.index = adata_obs.index.astype(str)

adata.obs = adata_obs

In [ ]:
# adding in the pathway PROGENy estimates
#adata.obs = pd.concat([adata.obs,pathway_matrix],axis=1)

In [ ]:
clustering_keys = {
    'SpatialFusion (UNI+scGPT)': 'leiden_gcn_uni_scgpt',
    'SpatialFusion (UNI+Nicheformer)': 'leiden_gcn_uni_nicheformer',
    'SpatialFusion (Virchow+scGPT)': 'leiden_gcn_virchow_scgpt',
    'SpatialFusion (Virchow+Nicheformer)': 'leiden_gcn_virchow_nicheformer',
    'SpatialFusion (UNI+scGPT, finetuned)': 'leiden_gcn_finetuned_uni_scgpt',
    'SpatialFusion (UNI+scGPT, H&E)': 'leiden_gcn_he',
    'NicheCompass': 'leiden_nichecompass',
    'BANKSY': 'leiden_banksy_08',
    'Nicheformer': 'leiden_nicheformer',
    'scGPT-spatial': 'leiden_scgptspatial',
    'OmiCLIP text': 'leiden_omiclip_text',
    'OmiCLIP image': 'leiden_omiclip_image',
    'Novae': 'leiden_novae',
    'Scanpy': 'leiden_scanpy',
}

results_df = compute_all_metrics(adata, clustering_keys)


In [ ]:
plot_benchmark_heatmap(results_df, title="BRCA Benchmark", savefig='../../../SpatialFusion/results/figures_Fig3/BRCA_benchmark_afterreview.svg')

## Plot the clusters in space

In [ ]:
def plot_spatial_clusters_panel(
    adata,
    method_mapping,
    color_dict,
    coord_keys=("X_coord", "Y_coord"),
    ncols=5,
    savefig=None,
    rasterize_points=True,
):
    """
    Plot spatial clustering panels for multiple methods in a Nature Genetics style.
    """

    x_key, y_key = coord_keys
    method_keys = list(method_mapping.values())
    method_titles = list(method_mapping.keys())
    n_methods = len(method_keys)
    nrows = math.ceil(n_methods / ncols)

    # --- Shared color palette across all cluster IDs (as strings) ---
    all_labels = np.unique(
        np.concatenate([
            adata.obs[k].astype(str).values for k in method_keys
        ])
    )

    # --- Figure style ---
    sns.set_style("white")
    sns.set_context("talk", font_scale=1.3)

    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols, figsize=(5.5 * ncols, 5 * nrows), dpi=300
    )
    axes = np.array(axes).reshape(-1)

    for i, (display_name, method) in enumerate(method_mapping.items()):
        ax = axes[i]

        # --- Convert hue column to string to match color_dict keys ---
        hue_values = adata.obs[method].astype(str)

        sns.scatterplot(
            x=adata.obs[x_key],
            y=adata.obs[y_key],
            hue=hue_values,
            palette=color_dict[method],
            s=1,
            linewidth=0,
            alpha=0.9,
            ax=ax,
            legend=False,
            rasterized=rasterize_points,
        )

        ax.invert_yaxis()
        ax.set_aspect("equal")

        # --- Titles & styling ---
        ax.set_title(display_name, fontsize=14, fontweight="normal", pad=10)
        ax.set_xlabel("", fontsize=18, labelpad=8, fontweight="normal")
        ax.set_ylabel("", fontsize=18, labelpad=8, fontweight="normal")
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ["top", "right", "left", "bottom"]:
            ax.spines[spine].set_visible(False)
        ax.grid(False)

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()

    # --- Shared legend ---
    rep_method = list(color_dict.keys())[0]
    rep_palette = color_dict[rep_method]
    
    handles = []
    for label in sorted(all_labels):
        if label in rep_palette:
            handles.append(
                plt.Line2D(
                    [0], [0],
                    marker="o",
                    color="none",
                    markerfacecolor=rep_palette[label],
                    markersize=8,
                    label=label,
                )
            )
    
    legend_fig = plt.figure(figsize=(2.5, 0.4 * len(handles)), dpi=300)
    legend_fig.legend(
        handles=handles,
        loc="center",
        title="Cluster",
        frameon=False,
        ncol=1,
        fontsize=12,
        title_fontsize=14,
    )
    legend_fig.tight_layout()
    
    if savefig:
        fig.savefig(
            f"{savefig}_panel.png",
            dpi=200,
            bbox_inches="tight",
            transparent=True,
        )
        legend_fig.savefig(
            f"{savefig}_legend.svg",
            dpi=300,
            bbox_inches="tight",
            transparent=True,
        )
        print(f"Saved: {savefig}_panel.png and {savefig}_legend.svg")
    
    plt.show()
    plt.close(legend_fig)

In [ ]:
leiden_cols = [
       'leiden_gcn_uni_scgpt', 'leiden_gcn_uni_nicheformer',
       'leiden_gcn_virchow_scgpt', 'leiden_gcn_virchow_nicheformer', 'leiden_gcn_finetuned_uni_scgpt',
   'leiden_gcn_he', 'leiden_banksy_08',
       'leiden_nichecompass', 'leiden_nicheformer', 'leiden_scgptspatial',
        'leiden_omiclip_text',
       'leiden_omiclip_image', 'leiden_novae', 'leiden_scanpy', ]

In [ ]:
palette_specs = {
            l: tab20_filtered for l in leiden_cols
        }

palette_dict_2 = build_palettes_from_adata(adata, palette_specs)

In [ ]:
method_keys = list(clustering_keys.values())

plot_spatial_clusters_panel(
    adata,
    color_dict=palette_dict_2,
    method_mapping=clustering_keys,
    ncols=6,
    savefig='../../../SpatialFusion/results/figures_Fig3/panel_viz_clusters_afterreview.svg',
)
